# HepGPU Demo: 2D No-Barrier Case

## Installation

In [ ]:

!pip install git+https://github.com/navasmontilla/HepGPU.git
!pip install zarr numpy pyevtk SimpleITK scipy pyvista panel ipywidgets

## Imports

In [ ]:

import numpy as np
import os
from HepGPU.core import run
from HepGPU.zarr_utils import export_zarr_to_vtk, export_zarr_to_png, export_zarr_to_2d_snapshot
import pyvista as pv
import panel as pn
import glob
import io
import matplotlib.pyplot as plt
from IPython.display import Image, display


## Grid

In [ ]:

nx, ny, nz = 101, 101, 3
Lx = Ly = Lz = 1.0

x = np.linspace(0, Lx, nx)
y = np.linspace(0, Ly, ny)
z = np.linspace(0, Lz, nz)

X, Y, Z = np.meshgrid(x, y, z, indexing="ij")


## Geometry and masks

In [ ]:

xc, yc = 0.0, 0.0

R_xy = np.sqrt((X - xc)**2 + (Y - yc)**2)

R_inflow = 0.21 * min(Lx, Ly, Lz)
mask_inflow = R_xy <= R_inflow

masks = None


## Model parameters

In [ ]:

params = {
    "a5": 0.06,
    "a3": 0.5,
    "dTc": 0.1,
    "d1": 0.01,
    "dTh": 0.5,
    "d3": 0.06,
}

out_name = "output_2d_no_barrier.zarr"
results_dir = "results_2d_no_barrier"


## Run simulation

In [ ]:

run(
    use_gpu=False,
    Lx=Lx,
    Ly=Ly,
    Lz=Lz,

    nx=nx,
    ny=ny,
    nz=nz,

    tf=400,
    td=1,
    
    sigma=0.15,
    params=params,
    mask_inflow=mask_inflow,
    CI_values=(0.1, 0.0, 0.0, 0.0),
    masks=masks,
    out_name=out_name
)


## Export results

In [ ]:

export_zarr_to_vtk(
    zarr_file=out_name,
    output_dir=os.path.join(results_dir, "vtk")
)

export_zarr_to_png(
    zarr_file=out_name,
    output_dir=os.path.join(results_dir, "plots")
)

export_zarr_to_2d_snapshot(
    zarr_file=out_name,
    output_path=os.path.join(results_dir, "spatial_snapshot_step_400.png"),
    step=400,
)

print("Export completed.")


## Visualization

In [ ]:
img_path = os.path.join(results_dir, "spatial_snapshot_step_400.png")
if os.path.exists(img_path):
    display(Image(filename=img_path))
else:
    print("Snapshot image not found.")

In [ ]:

gif_path = os.path.join(results_dir, "plots", "animation.gif")

if os.path.exists(gif_path):
    display(Image(filename=gif_path))
else:
    print("No GIF found.")


## Mini ParaView (Colab image version)

In [ ]:
pn.extension()
pv.OFF_SCREEN = True

files = sorted(glob.glob(os.path.join(results_dir, "vtk", "*.vti")))
if len(files) == 0:
    raise ValueError("No .vti files found in the VTK output folder.")

grid0 = pv.read(files[0])
scalars_list = grid0.array_names

# ---------------------------------
# Widgets
# ---------------------------------
time_slider = pn.widgets.IntSlider(name="Time Step", start=0, end=len(files)-1, value=0)

scalar_select = pn.widgets.Select(name="Variable", options=scalars_list, value=scalars_list[0])

cmap_select = pn.widgets.Select(
    name="Colormap",
    options=["viridis", "plasma", "coolwarm", "inferno"],
    value="viridis"
)

mode_select = pn.widgets.Select(
    name="View Mode",
    options=["surface", "slice", "contour", "threshold"],
    value="surface"
)

view_select = pn.widgets.Select(
    name="Camera View",
    options=["isometric", "xy", "xz", "yz"],
    value="isometric"
)

show_box_checkbox = pn.widgets.Checkbox(name="Show Box", value=True)

n_contours_slider = pn.widgets.IntSlider(name="N Contours", start=1, end=10, value=5)

slice_x_slider = pn.widgets.FloatSlider(name="Slice X", start=0, end=1, step=0.01, value=0.5)
slice_y_slider = pn.widgets.FloatSlider(name="Slice Y", start=0, end=1, step=0.01, value=0.5)
slice_z_slider = pn.widgets.FloatSlider(name="Slice Z", start=0, end=1, step=0.01, value=0.5)

# transparencia
opacity_slider = pn.widgets.FloatSlider(
    name="Opacity",
    start=0.05,
    end=1.0,
    step=0.05,
    value=1.0
)

# threshold
threshold_slider = pn.widgets.FloatSlider(
    name="Threshold (min)",
    start=0.0,
    end=1.0,
    step=0.01,
    value=0.1
)

# ---------------------------------
# Render
# ---------------------------------
def render_image(time_step, scalar, cmap, mode, view, show_box,
                 n_contours, sx, sy, sz, opacity, threshold_val):

    grid = pv.read(files[time_step])
    plotter = pv.Plotter(off_screen=True, window_size=(800, 500))

    xmin, xmax, ymin, ymax, zmin, zmax = grid.bounds
    x0 = xmin + sx * (xmax - xmin)
    y0 = ymin + sy * (ymax - ymin)
    z0 = zmin + sz * (zmax - zmin)

    # ---------------------------------
    # MODOS
    # ---------------------------------
    if mode == "surface":
        plotter.add_mesh(grid, scalars=scalar, cmap=cmap, opacity=opacity)

    elif mode == "slice":
        sliced = grid.slice_orthogonal(x=x0, y=y0, z=z0)
        plotter.add_mesh(sliced, scalars=scalar, cmap=cmap, opacity=opacity)

    elif mode == "contour":
        try:
            contour = grid.contour(isosurfaces=n_contours, scalars=scalar)
            plotter.add_mesh(contour, scalars=scalar, cmap=cmap, opacity=opacity)
        except Exception as e:
            plotter.add_text(f"Contour failed: {e}", font_size=10)

    # threshold
    elif mode == "threshold":
        try:
            th = grid.threshold(value=threshold_val, scalars=scalar)
            plotter.add_mesh(th, scalars=scalar, cmap=cmap, opacity=opacity)
        except Exception as e:
            plotter.add_text(f"Threshold failed: {e}", font_size=10)

    # ---------------------------------
    if show_box:
        plotter.add_mesh(grid.outline(), color="white", line_width=2)

    if view == "xy":
        plotter.view_xy()
    elif view == "xz":
        plotter.view_xz()
    elif view == "yz":
        plotter.view_yz()
    else:
        plotter.view_isometric()

    plotter.reset_camera()

    img = plotter.screenshot(return_img=True)
    plotter.close()

    buf = io.BytesIO()
    plt.imsave(buf, img)
    buf.seek(0)

    return pn.pane.PNG(buf.getvalue(), sizing_mode="stretch_width")

# ---------------------------------
# Binding
# ---------------------------------
viewer = pn.bind(
    render_image,
    time_slider,
    scalar_select,
    cmap_select,
    mode_select,
    view_select,
    show_box_checkbox,
    n_contours_slider,
    slice_x_slider,
    slice_y_slider,
    slice_z_slider,
    opacity_slider,
    threshold_slider
)

# ---------------------------------
# Layout
# ---------------------------------
dashboard = pn.Column(
    "# Mini ParaView",
    pn.Row(time_slider),
    pn.Row(scalar_select, cmap_select),
    pn.Row(mode_select, view_select),
    pn.Row(show_box_checkbox, n_contours_slider),
    pn.Row(opacity_slider, threshold_slider),
    pn.Row(slice_x_slider, slice_y_slider, slice_z_slider),
    viewer,
)

dashboard